# Lab Task 9 - Fine-Tuning YOLO on a Custom Dataset

Computer Vision  
Name: Sarim Azhar  
Roll No: 247076

**Dataset:** Medical Pills (Ultralytics dataset hub) - 1 class `pill`, 92 training images and 23 validation images. Pills are not one of the 80 COCO classes, so the pretrained model cannot detect them before fine-tuning.

## Step 0: Install Ultralytics

In [ ]:
!pip install -q ultralytics

import ultralytics
ultralytics.checks()

## Step 1: Dataset

The dataset is in YOLO format and is downloaded automatically by Ultralytics using `medical-pills.yaml`. The yaml looks like this:

```yaml
path: ../datasets/medical-pills
train: images/train
val: images/val

names:
  0: pill
```

In [ ]:
from ultralytics.utils.downloads import download
import os

# download the dataset into /content/datasets/medical-pills
download("https://github.com/ultralytics/assets/releases/download/v0.0.0/medical-pills.zip", dir="/content/datasets")

for root, dirs, files in os.walk("/content/datasets/medical-pills"):
    print(root, "->", len(files), "files")

## Step 2: Train

I used **yolo11n.pt** (the nano model) because it is the smallest YOLO11 model and the session is running on CPU, so a bigger model would take too long to train. The dataset also has only 92 training images and one easy class, so the extra capacity of a larger model is not needed.

Settings: `imgsz=320` instead of 640 to keep CPU training time low, `batch=8` because CPU RAM is limited, `epochs=20` which is enough for a single-class dataset, and `patience=5` to stop early if the model stops improving.

In [ ]:
from ultralytics import YOLO

# start from pretrained COCO weights (transfer learning)
model = YOLO("yolo11n.pt")

results = model.train(
    data="medical-pills.yaml",
    epochs=20,
    imgsz=320,
    batch=8,
    patience=5,
    device="cpu",
    workers=2,
)

## Step 3: Validate

Running `model.val()` to get precision, recall, mAP50 and mAP50-95 on the validation set.

In [ ]:
metrics = model.val()

print("Precision :", metrics.box.mp)
print("Recall    :", metrics.box.mr)
print("mAP50     :", metrics.box.map50)
print("mAP50-95  :", metrics.box.map)

**Results:**

| Metric | Value |
|---|---|
| Precision | |
| Recall | |
| mAP50 | |
| mAP50-95 | |

(fill in from the output above)

Precision tells how many of the predicted pills were actually pills, recall tells how many of the real pills the model found, mAP50 is the average precision at IoU 0.50, and mAP50-95 is the stricter average over IoU 0.50 to 0.95.

## Step 4: Predict on test images

Running the fine-tuned model on 3 validation images, which were not used during training.

In [ ]:
import glob
import matplotlib.pyplot as plt
from ultralytics import YOLO

# path to the best weights saved during training
best_weights = model.trainer.best
print("Weights:", best_weights)

my_model = YOLO(best_weights)

test_images = sorted(glob.glob("/content/datasets/medical-pills/images/val/*.jpg"))[:3]
print("Test images:", test_images)

for img in test_images:
    result = my_model(img)[0]
    plt.figure(figsize=(8, 8))
    plt.imshow(result.plot()[..., ::-1])
    plt.axis("off")
    plt.title(img.split("/")[-1])
    plt.show()
    print("Objects detected:", len(result.boxes))

## Conclusion

The pretrained YOLO11n model was fine-tuned on the Medical Pills dataset so that it can detect pills, which is not one of the 80 COCO classes. After training, the model was validated and tested on three unseen images, and it correctly drew bounding boxes around the pills.